# 06 - ChromaDB Retrieval

## Definition
A vector database stores embeddings and supports efficient nearest-neighbor retrieval with metadata filtering.

## Theory
Approximate nearest-neighbor indexing enables scalable semantic retrieval.

## Motivation
Without a vector DB, large-scale dense retrieval becomes too slow and operationally fragile.

## Architecture
Chunks + embeddings -> Chroma collection -> query embedding -> top-k results.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [ ]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [ ]:
bundle = runner.ingest_dataset()
chunking = runner.run_chunking(bundle['documents'][:650], strategy=ChunkingStrategy.RECURSIVE)
runner.index_chunks(chunking.chunks, reset=True)

query = 'What is the main argument in this passage?'
hits = runner.retrieval_engine.query(query=query, top_k=6, dedupe_by_doc=True)

pd.DataFrame([
    {
        'rank': i + 1,
        'doc_id': h.doc_id,
        'score': h.score,
        'title': h.metadata.get('title', 'unknown'),
    }
    for i, h in enumerate(hits)
])


## Comparison: ChromaDB vs FAISS vs Pinecone vs Weaviate vs Qdrant
- ChromaDB: local-first simplicity and beginner-friendly persistence
- FAISS: library-level ANN speed, fewer database features
- Pinecone: managed vector DB, stronger ops but paid
- Weaviate/Qdrant: production vector DB features with richer deployment patterns